# Build Two Multitask Datasets
Creates two datasets:
1. **Fully labeled dataset**: All classification images have masks, all segmentation images have labels
2. **Partially labeled dataset**: Half of each subset has both labels and masks, half has only one

In [1]:
# Common imports and path configuration
from pathlib import Path
import random
import shutil

import pandas as pd

random.seed(42)
project_root = Path("/Volumes/T7/largeProject_copy")
dataset_root = project_root / "dataset"
dataset_manifest_path = dataset_root / "manifest.csv"

# Output directories for the two derived multitask datasets
output_fully_labeled = project_root / "multitask_fully_labeled"
output_partially_labeled = project_root / "multitask_partially_labeled"

splits = ("train", "val", "test")
class_columns = ["MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC"]

print(f"Source dataset manifest: {dataset_manifest_path}")
print(f"Fully labeled output:    {output_fully_labeled}")
print(f"Partially labeled output:{output_partially_labeled}")


Source dataset manifest: /Volumes/T7/largeProject_copy/dataset/manifest.csv
Fully labeled output:    /Volumes/T7/largeProject_copy/multitask_fully_labeled
Partially labeled output:/Volumes/T7/largeProject_copy/multitask_partially_labeled


In [2]:
# (Re)build dataset/manifest.csv from the current filesystem state
# Walks dataset/{classification,segmentation}/{train,val,test}/ and consults
#   - per-split classification CSVs in classification/<split>/ground_truth/
#   - utilities/seg_labels.csv  (optional, for per-image segmentation labels)
#   - generated_masks/<split>/   (optional, supplies masks for classification images)
# Run this whenever utilities/seg_labels.csv or generated_masks/ changes so the
# manifest reflects the new data. Splits stay aligned with the on-disk folders.
print("\n🔨 (Re)building dataset/manifest.csv from current filesystem...")

classification_root = dataset_root / "classification"
segmentation_root = dataset_root / "segmentation"
generated_masks_root = project_root / "generated_masks"
seg_labels_csv = project_root / "utilities" / "seg_labels.csv"

manifest_rows = []
seen_ids: set = set()

# 1) Classification images: walk per-split input/, look up labels in per-split CSV
for split in splits:
    input_dir = classification_root / split / "input"
    gt_dir = classification_root / split / "ground_truth"
    if not input_dir.exists():
        continue
    csv_files = sorted(p for p in gt_dir.glob("*.csv"))
    if not csv_files:
        print(f"  WARN: no classification CSV under {gt_dir}")
        continue
    cls_df = pd.read_csv(csv_files[0])
    if "image" not in cls_df.columns:
        raise RuntimeError(f"Classification CSV {csv_files[0]} missing 'image' column")
    missing_cls_cols = [c for c in class_columns if c not in cls_df.columns]
    if missing_cls_cols:
        raise RuntimeError(f"{csv_files[0]} missing class columns: {missing_cls_cols}")
    cls_lookup = cls_df.drop_duplicates(subset=["image"]).set_index("image")

    for jpg in sorted(input_dir.glob("*.jpg")):
        image_id = jpg.stem
        if image_id in seen_ids or image_id not in cls_lookup.index:
            continue
        labels = [int(float(cls_lookup.at[image_id, col])) for col in class_columns]
        gen_mask = generated_masks_root / split / f"{image_id}_segmentation.png"
        rel_image = jpg.relative_to(project_root)
        rel_mask = gen_mask.relative_to(project_root) if gen_mask.exists() else None
        row = {
            "image_id": image_id,
            "image": str(rel_image),
            "mask": str(rel_mask) if rel_mask is not None else None,
            "split": split,
            "source": "classification",
        }
        for i, col in enumerate(class_columns):
            row[col] = labels[i]
        manifest_rows.append(row)
        seen_ids.add(image_id)

# 2) Segmentation labels lookup (optional)
seg_labels_by_id: dict = {}
if seg_labels_csv.exists():
    seg_df = pd.read_csv(seg_labels_csv)
    if "image" not in seg_df.columns:
        print(f"  WARN: {seg_labels_csv} missing 'image' column; ignoring")
    else:
        missing_seg_cols = [c for c in class_columns if c not in seg_df.columns]
        if missing_seg_cols:
            print(f"  WARN: {seg_labels_csv} missing class columns {missing_seg_cols}; segmentation rows will have empty labels")
        else:
            for r in seg_df.itertuples(index=False):
                rel = Path(str(r.image))
                image_id = rel.stem
                seg_labels_by_id[image_id] = [int(float(getattr(r, col))) for col in class_columns]
else:
    print(f"  NOTE: {seg_labels_csv} not found; segmentation rows will have empty labels")

# 3) Segmentation images: walk per-split input/ + ground_truth/, attach labels if known
for split in splits:
    input_dir = segmentation_root / split / "input"
    mask_dir = segmentation_root / split / "ground_truth"
    if not input_dir.exists() or not mask_dir.exists():
        continue
    for jpg in sorted(input_dir.glob("*.jpg")):
        image_id = jpg.stem
        if image_id in seen_ids:
            continue  # already covered by the classification side
        mask_path = mask_dir / f"{image_id}_segmentation.png"
        if not mask_path.exists():
            continue
        labels = seg_labels_by_id.get(image_id)
        rel_image = jpg.relative_to(project_root)
        rel_mask = mask_path.relative_to(project_root)
        row = {
            "image_id": image_id,
            "image": str(rel_image),
            "mask": str(rel_mask),
            "split": split,
            "source": "segmentation",
        }
        for i, col in enumerate(class_columns):
            row[col] = None if labels is None else labels[i]
        manifest_rows.append(row)
        seen_ids.add(image_id)

manifest_df = pd.DataFrame(manifest_rows)
dataset_root.mkdir(parents=True, exist_ok=True)
manifest_df.to_csv(dataset_manifest_path, index=False)

print(f"✅ Wrote {dataset_manifest_path}")
print(f"   Total rows: {len(manifest_df)}")
for s in splits:
    n = int((manifest_df["split"] == s).sum())
    pct = n / len(manifest_df) if len(manifest_df) else 0.0
    print(f"   {s:5s}: {n:6d} ({pct:.2%})")
print(f"   classification rows: {int((manifest_df['source'] == 'classification').sum())}")
print(f"   segmentation rows:   {int((manifest_df['source'] == 'segmentation').sum())}")
print(f"   classification with mask: {int(((manifest_df['source'] == 'classification') & manifest_df['mask'].notna()).sum())}")



🔨 (Re)building dataset/manifest.csv from current filesystem...
✅ Wrote /Volumes/T7/largeProject_copy/dataset/manifest.csv
   Total rows: 15414
   train:  10790 (70.00%)
   val  :   2312 (15.00%)
   test :   2312 (15.00%)
   classification rows: 11720
   segmentation rows:   3694
   classification with mask: 11720


In [3]:
# Load every record from the source dataset manifest
# The manifest already encodes the 70/15/15 split, the modality (`source`),
# the relative paths to image and (optional) mask, and the class labels for
# samples that have them. We split rows back into the same `classification_records`
# and `segmentation_entries` structures the build cells below expect.
if not dataset_manifest_path.exists():
    raise FileNotFoundError(
        f"Source manifest not found at {dataset_manifest_path}. "
        "Build it first (or restore it) before running this notebook."
    )

dataset_df = pd.read_csv(dataset_manifest_path)

required_cols = {"image_id", "image", "mask", "split", "source"} | set(class_columns)
missing = required_cols - set(dataset_df.columns)
if missing:
    raise RuntimeError(f"Manifest missing required columns: {sorted(missing)}")

classification_records = []
segmentation_entries = []

for row in dataset_df.itertuples(index=False):
    image_id = str(row.image_id)
    image_path = project_root / str(row.image)

    raw_mask = getattr(row, "mask")
    has_mask_path = isinstance(raw_mask, str) and raw_mask.strip() != "" and raw_mask != "nan"
    mask_path = project_root / raw_mask if has_mask_path else None

    label_values = [getattr(row, col) for col in class_columns]
    has_labels = all(pd.notna(v) for v in label_values)
    labels = [int(float(v)) for v in label_values] if has_labels else None

    common = {
        "image_id": image_id,
        "image_path": image_path,
        "mask_path": mask_path,
        "split": str(row.split),
        "labels": labels,
    }
    if str(row.source) == "classification":
        classification_records.append({**common, "has_mask": has_mask_path})
    else:
        segmentation_entries.append({**common, "has_labels": has_labels})

print(f"\n📊 Loaded from manifest:")
print(f"   Total rows: {len(dataset_df)}")
print(f"   Classification records: {len(classification_records)}")
print(f"     - with masks: {sum(1 for r in classification_records if r['has_mask'])}")
print(f"   Segmentation entries:   {len(segmentation_entries)}")
print(f"     - with labels: {sum(1 for e in segmentation_entries if e['has_labels'])}")
print(f"   Split distribution:")
for s in splits:
    n = int((dataset_df["split"] == s).sum())
    pct = n / len(dataset_df) if len(dataset_df) else 0.0
    print(f"     {s:5s}: {n:6d} ({pct:.2%})")



📊 Loaded from manifest:
   Total rows: 15414
   Classification records: 11720
     - with masks: 11720
   Segmentation entries:   3694
     - with labels: 3694
   Split distribution:
     train:  10790 (70.00%)
     val  :   2312 (15.00%)
     test :   2312 (15.00%)


In [4]:
# Build Fully Labeled Dataset
# All classification images with generated masks + All segmentation images with labels
print("\n🔨 Building Fully Labeled Dataset...")

fully_labeled_records = []

# Add classification images that have generated masks
for record in classification_records:
    if not record["has_mask"]:
        continue  # Skip images without masks
    fully_labeled_records.append({
        "image_id": record["image_id"],
        "image_path": record["image_path"],
        "mask_path": record["mask_path"],
        "split": record["split"],
        "labels": record["labels"],
        "source": "classification"
    })

# Add segmentation images that have labels
for entry in segmentation_entries:
    if not entry["has_labels"]:
        continue  # Skip images without labels
    fully_labeled_records.append({
        "image_id": entry["image_id"],
        "image_path": entry["image_path"],
        "mask_path": entry["mask_path"],
        "split": entry["split"],
        "labels": entry["labels"],
        "source": "segmentation"
    })

# Create directory structure and copy files
output_fully_labeled.mkdir(parents=True, exist_ok=True)
(output_fully_labeled / "images").mkdir(exist_ok=True)
(output_fully_labeled / "masks").mkdir(exist_ok=True)

manifest_rows = []
for record in fully_labeled_records:
    # Copy image
    dst_image = output_fully_labeled / "images" / f"{record['image_id']}.jpg"
    if not dst_image.exists():
        shutil.copy2(record["image_path"], dst_image)
    
    # Copy mask
    dst_mask = output_fully_labeled / "masks" / f"{record['image_id']}_segmentation.png"
    if not dst_mask.exists():
        shutil.copy2(record["mask_path"], dst_mask)
    
    # Build manifest row
    manifest_row = {
        "image_id": record["image_id"],
        "image": f"images/{record['image_id']}.jpg",
        "mask": f"masks/{record['image_id']}_segmentation.png",
        "split": record["split"],
        "source": record["source"]
    }
    for i, col in enumerate(class_columns):
        manifest_row[col] = record["labels"][i]
    manifest_rows.append(manifest_row)

# Save manifest
manifest_df = pd.DataFrame(manifest_rows)
manifest_df.to_csv(output_fully_labeled / "manifest.csv", index=False)

print(f"✅ Fully Labeled Dataset Created:")
print(f"   Total samples: {len(manifest_rows)}")
print(f"   From classification: {sum(1 for r in manifest_rows if r['source'] == 'classification')}")
print(f"   From segmentation: {sum(1 for r in manifest_rows if r['source'] == 'segmentation')}")
print(f"   Train: {sum(1 for r in manifest_rows if r['split'] == 'train')}")
print(f"   Val: {sum(1 for r in manifest_rows if r['split'] == 'val')}")
print(f"   Test: {sum(1 for r in manifest_rows if r['split'] == 'test')}")


🔨 Building Fully Labeled Dataset...
✅ Fully Labeled Dataset Created:
   Total samples: 15414
   From classification: 11720
   From segmentation: 3694
   Train: 10790
   Val: 2312
   Test: 2312


In [6]:
# Sanity-check the 70/15/15 split across all three datasets
# Verifies:
#   1. Each dataset's split distribution is close to 70/15/15.
#   2. The same image_id is assigned the same split across datasets.
print("\n🔎 Sanity-checking splits...")

TOLERANCE = 0.01  # acceptable absolute deviation from target ratios
TARGETS = {"train": 0.70, "val": 0.15, "test": 0.15}


def _check_ratios(name: str, split_series) -> bool:
    counts = split_series.value_counts()
    total = int(counts.sum())
    print(f"\n  {name} (n={total}):")
    ok = True
    for s in splits:
        count = int(counts.get(s, 0))
        ratio = count / total if total else 0.0
        delta = ratio - TARGETS[s]
        flag = "OK" if abs(delta) <= TOLERANCE else "OFF"
        if flag == "OFF":
            ok = False
        print(f"    {s:5s}: {count:6d}  ({ratio:6.2%}, target {TARGETS[s]:.0%}, Δ={delta:+.2%}) [{flag}]")
    return ok


# 1) source dataset/manifest.csv
ok_dataset = _check_ratios("dataset/manifest.csv", dataset_df["split"])

# 2) multitask_fully_labeled/manifest.csv
ful_df = pd.read_csv(output_fully_labeled / "manifest.csv")
ok_full = _check_ratios("multitask_fully_labeled/manifest.csv", ful_df["split"])

# 3) multitask_partially_labeled — combine all three sub-manifests
part_dfs = []
for sub in ("paired", "classification_only", "segmentation_only"):
    p = output_partially_labeled / sub / "manifest.csv"
    if p.exists():
        part_dfs.append(pd.read_csv(p))
part_df = pd.concat(part_dfs, ignore_index=True) if part_dfs else pd.DataFrame()
ok_part = (
    _check_ratios("multitask_partially_labeled (combined)", part_df["split"])
    if not part_df.empty
    else False
)

# 4) Cross-dataset image_id -> split consistency
print("\n  cross-dataset split consistency (per image_id):")
sources = {
    "dataset": dataset_df[["image_id", "split"]],
    "fully_labeled": ful_df[["image_id", "split"]],
    "partially_labeled": part_df[["image_id", "split"]] if not part_df.empty else None,
}
merged = None
for name, df in sources.items():
    if df is None:
        continue
    df = df.drop_duplicates(subset=["image_id"]).rename(columns={"split": f"split_{name}"})
    merged = df if merged is None else merged.merge(df, on="image_id", how="outer")

split_cols = [c for c in merged.columns if c.startswith("split_")]
def _disagrees(row):
    vals = {row[c] for c in split_cols if isinstance(row[c], str)}
    return len(vals) > 1
mismatches = merged[merged.apply(_disagrees, axis=1)]
if mismatches.empty:
    print("    OK — every image_id has a single split assignment across all datasets")
    ok_consistency = True
else:
    print(f"    OFF — {len(mismatches)} image_ids have inconsistent splits across datasets")
    print(mismatches.head(10).to_string(index=False))
    ok_consistency = False

print("\n" + "=" * 60)
all_ok = ok_dataset and ok_full and ok_part and ok_consistency
print(f"OVERALL: {'✅ ALL CHECKS PASSED' if all_ok else '❌ SOME CHECKS FAILED'}")
print("=" * 60)



🔎 Sanity-checking splits...

  dataset/manifest.csv (n=15414):
    train:  10790  (70.00%, target 70%, Δ=+0.00%) [OK]
    val  :   2312  (15.00%, target 15%, Δ=-0.00%) [OK]
    test :   2312  (15.00%, target 15%, Δ=-0.00%) [OK]

  multitask_fully_labeled/manifest.csv (n=15414):
    train:  10790  (70.00%, target 70%, Δ=+0.00%) [OK]
    val  :   2312  (15.00%, target 15%, Δ=-0.00%) [OK]
    test :   2312  (15.00%, target 15%, Δ=-0.00%) [OK]

  multitask_partially_labeled (combined) (n=15414):
    train:  10790  (70.00%, target 70%, Δ=+0.00%) [OK]
    val  :   2312  (15.00%, target 15%, Δ=-0.00%) [OK]
    test :   2312  (15.00%, target 15%, Δ=-0.00%) [OK]

  cross-dataset split consistency (per image_id):
    OK — every image_id has a single split assignment across all datasets

OVERALL: ✅ ALL CHECKS PASSED
